<a href="https://colab.research.google.com/github/BagaskaraAdhi/UTSTextMining/blob/main/2318090BagaskaraAdhiPradanatrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.6 MB/s eta 0:00:00


In [5]:
import json
import os
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import numpy as np
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from google.colab import drive

# ─── 1. MOUNT DRIVE ───────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=True)

# ─── 2. BASE PATH ─────────────────────────────────────────────────────────────
base_path = '/content/drive/MyDrive/Text Mining/UTS/Test/'
data_json_path = base_path + "Data.json"  # huruf D kapital sesuai nama file di Drive

print(f"✅ Folder: {base_path}")

# ─── 3. CEK KEBERADAAN Data.json ──────────────────────────────────────────────
if not os.path.exists(data_json_path):
    raise FileNotFoundError(
        f"❌ File tidak ditemukan: {data_json_path}\n"
        f"   Pastikan file 'Data.json' sudah ada di folder Test di Google Drive."
    )

print("✅ Data.json ditemukan!")

# ─── 4. LOAD INDONESIAN STEMMER ───────────────────────────────────────────────
print("\n🔄 Memuat stemmer...")
factory = StemmerFactory()
stemmer = factory.create_stemmer()
print("✅ Stemmer siap!")

# ─── 5. FUNGSI PREPROCESSING ──────────────────────────────────────────────────
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    stemmed_words = [stemmer.stem(word) for word in words]
    return ' '.join(stemmed_words)

# ─── 6. LOAD DATA ─────────────────────────────────────────────────────────────
print("\n🔄 Memuat Data.json...")
with open(data_json_path, "r", encoding="utf-8") as file:
    corpus = json.load(file)["qa_corpus"]

questions = [item["question"] for item in corpus]
answers   = [item["answer"]   for item in corpus]

print(f"✅ Data dimuat: {len(corpus)} pasang QA")

# ─── 7. PREPROCESSING ─────────────────────────────────────────────────────────
print("\n🔄 Preprocessing teks...")
preprocessed_questions = [preprocess_text(q) for q in questions]
preprocessed_answers   = [preprocess_text(a) for a in answers]
combined_corpus        = preprocessed_questions + preprocessed_answers
print("✅ Preprocessing selesai!")

# ─── 8. TRAINING TF-IDF ───────────────────────────────────────────────────────
print("\n🔄 Training TF-IDF vectorizer...")
vectorizer = TfidfVectorizer()
vectorizer.fit(combined_corpus)
print(f"✅ Vectorizer dilatih dengan {len(vectorizer.vocabulary_)} kata unik")

# ─── 9. SIMPAN VECTORIZER ─────────────────────────────────────────────────────
vectorizer_path = base_path + "vectorizer.joblib"
joblib.dump(vectorizer, vectorizer_path)
print(f"✅ vectorizer.joblib disimpan di: {vectorizer_path}")

# ─── 10. TRANSFORM PERTANYAAN KE VEKTOR ───────────────────────────────────────
print("\n🔄 Mentransform pertanyaan ke vektor...")
question_vectors = vectorizer.transform(preprocessed_questions)
print("✅ Transform selesai!")

# ─── 11. SIMPAN KE vector.json ────────────────────────────────────────────────
print("\n🔄 Menyimpan vector.json...")
vector_data = []
for question, answer, vector in zip(questions, answers, question_vectors):
    vector_data.append({
        "question": question,
        "answer":   answer,
        "vector":   vector.toarray().tolist()[0]
    })

vector_json_path = base_path + "vector.json"
with open(vector_json_path, "w", encoding="utf-8") as file:
    json.dump(vector_data, file, indent=4, ensure_ascii=False)

print(f"✅ vector.json disimpan di: {vector_json_path}")

# ─── 12. RINGKASAN HASIL ──────────────────────────────────────────────────────
print("\n" + "="*55)
print("🎉 Training selesai! File tersimpan di folder Test:")
print(f"   📄 Data.json       (input)")
print(f"   📄 vectorizer.joblib (output)")
print(f"   📄 vector.json       (output)")
print(f"\n   Total data : {len(corpus)} pasang QA")
print(f"   Vocab size : {len(vectorizer.vocabulary_)} kata unik")
print(f"   Dimensi    : {question_vectors.shape[1]} fitur TF-IDF")
print("="*55)

Mounted at /content/drive
✅ Folder: /content/drive/MyDrive/Text Mining/UTS/Test/
✅ Data.json ditemukan!

🔄 Memuat stemmer...
✅ Stemmer siap!

🔄 Memuat Data.json...
✅ Data dimuat: 20 pasang QA

🔄 Preprocessing teks...
✅ Preprocessing selesai!

🔄 Training TF-IDF vectorizer...
✅ Vectorizer dilatih dengan 220 kata unik
✅ vectorizer.joblib disimpan di: /content/drive/MyDrive/Text Mining/UTS/Test/vectorizer.joblib

🔄 Mentransform pertanyaan ke vektor...
✅ Transform selesai!

🔄 Menyimpan vector.json...
✅ vector.json disimpan di: /content/drive/MyDrive/Text Mining/UTS/Test/vector.json

🎉 Training selesai! File tersimpan di folder Test:
   📄 Data.json       (input)
   📄 vectorizer.joblib (output)
   📄 vector.json       (output)

   Total data : 20 pasang QA
   Vocab size : 220 kata unik
   Dimensi    : 220 fitur TF-IDF
